# Customer request intent routing

Support queues receive short, ambiguous customer questions. A classifier that always chooses a category can route vague, sensitive or unrelated requests incorrectly.

Combine word and character TF-IDF features with a linear probabilistic classifier. Remove normalized duplicate conflicts and train/test overlap. Select a confidence/margin gate on validation, add lexical similarity and sensitive-request review checks, and return structured routing results with reference IDs.

Rebuild source artifacts with `python scripts/run_all.py` from the repository root. This walkthrough inspects the measured run.

In [1]:
from pathlib import Path
import json
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
filename = "metrics.json"
report = json.loads((root / "artifacts/routing" / filename).read_text(encoding="utf-8"))
print(json.dumps(report.get("test", report.get("decisions", report.get("aggregate"))), indent=2))

{
  "accuracy": 0.9039713541666666,
  "macro_f1": 0.9043299582488368,
  "auto_route_coverage": 0.8590494791666666,
  "auto_route_accuracy": 0.9586964759378552,
  "review_share": 0.14095052083333334,
  "auto_route_count": 2639
}


In [2]:
import pandas as pd
result = pd.read_csv(root / "artifacts/routing/holdout_predictions.csv")
print(result.head(8).to_string(index=False))

      actual              predicted  model_score  score_gate_pass
card_arrival    order_physical_card     0.111078            False
card_arrival           card_arrival     0.959537             True
card_arrival           card_arrival     0.400059             True
card_arrival           card_arrival     0.469769             True
card_arrival           card_arrival     0.758321             True
card_arrival card_delivery_estimate     0.692055             True
card_arrival           card_arrival     0.959758             True
card_arrival           card_arrival     0.495147             True


## Interpretation and limits

English BANKING77 only. Scores are not independently calibrated. Additional inference checks reduce routing coverage beyond the benchmark score gate. Regex redaction is partial. No generated answers, transformer fine-tuning, multilingual claim or production helpdesk integration.